# Google Gemini Chat Model


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


`langchain-google-genai` 4.x는 legacy `google-generativeai` SDK가 아니라 통합 `google-genai` SDK를 사용합니다. `GOOGLE_API_KEY`를 `.env`에 저장하고 코드에는 키를 넣지 않습니다.


In [ ]:
%pip install -qU 'langchain-google-genai>=4.0.0' python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

model_name = os.getenv("GEMINI_MODEL", "gemini-3.7-flash")
llm = ChatGoogleGenerativeAI(
    model=model_name,
    temperature=0,
    thinking_level="low",
)

for chunk in llm.stream("자연어 처리를 두 문장으로 설명해 주세요."):
    print(chunk.text, end="", flush=True)


## LCEL 체인


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "질문에 예 또는 아니오로 먼저 답하고 한 문장으로 설명하세요."),
        ("human", "{item}은 과일입니까?"),
    ]
)
chain = prompt | llm | StrOutputParser()

for text in chain.stream({"item": "사과"}):
    print(text, end="", flush=True)


## Safety settings

안전 설정을 낮추는 예시는 동작 설명용입니다. 실제 서비스에서는 사용 사례와 정책에 맞는 차단 수준을 유지합니다.


In [ ]:
from langchain_google_genai import HarmBlockThreshold, HarmCategory

safety_model = ChatGoogleGenerativeAI(
    model=model_name,
    temperature=0,
    safety_settings={
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    },
)


## Batch


In [ ]:
results = llm.batch(
    ["대한민국의 수도는?", "대한민국의 주요 관광지 5곳은?"],
    config={"max_concurrency": 2},
)
for result in results:
    print(result.text, "\n---")


## 이미지 입력

별도 `MultiModal` 보조 클래스 없이 표준 `HumanMessage` content block을 사용합니다. Gemini 3 계열은 `.content`가 블록 리스트일 수 있으므로 사용자에게 보여 줄 텍스트는 `.text`로 읽습니다.


In [ ]:
import base64
import mimetypes
from pathlib import Path
from langchain.messages import HumanMessage

image_path = Path("images/jeju-beach.jpg")
if not image_path.exists():
    raise FileNotFoundError(f"샘플 이미지를 준비하세요: {image_path.resolve()}")

mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
image_base64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")

message = HumanMessage(
    content=[
        {"type": "text", "text": "이 이미지를 소재로 짧은 시를 써 주세요."},
        {"type": "image", "base64": image_base64, "mime_type": mime_type},
    ]
)

response = llm.invoke([message])
print(response.text)
